In [ ]:
forget()
load("Library.sage")
from sage.geometry.triangulation.point_configuration import Triangulation

R.<a_51,a_41,a_31,a_21> = LaurentPolynomialRing(QQ);
R_poly = R.polynomial_ring();
Q.<t> = PolynomialRing(QQ);

LG = (a_51 * (1 + a_41) * (1 + a_41 + a_31) * (1 + a_41 + a_31 + a_21) * a_41^-1 * a_31^-1 * a_21^-1 + \
      (1 + a_41) * (1 + a_41 + a_31) * a_41^-1 * a_31^-1 * a_21^-1 + \
      (1 + a_41)^2 * a_41^-1 * a_31^-1 + 1) * (1 + a_41 + a_31 + a_21 + a_51^-1);

LG_mutated = (a_51^-1*a_41^-1*a_31^-2*a_21^-1) * \
((a_41*a_31) * ((a_21 + 1) * (a_31 + 1) + a_51) + 1) * \
((a_41*a_31) * (a_21 + 1) * (a_31 + 1) * (a_31*a_21 + a_51 + a_31 + a_21 + 1) + \
 (a_51*a_31*a_21 + (a_31 + 1) * (a_51 + a_21 + 1)));

LG_monomial_matrix = matrix(ZZ, [[0, 1, 1, 0],
                                 [0, 0, 1, 0],
                                 [0, 0, 0, 1],
                                 [1, 0, 0, 0]]);
LG_mutation_factor = a_31 + 1;
LG_monomial_change = GL_action(LG, LG_monomial_matrix);
#LG_mutated_alt = laurent_polynomial_mutation(LG_monomial_change, LG_mutation_factor);


## Newton polytope of the Landau--Ginzburg model

polytope = newton_polytope(LG_mutated);
polyhedron = Polyhedron(polytope.vertices(), base_ring=ZZ);

integral_points = polyhedron.integral_points();
interior_points = polytope.interior_points();

edges = polytope.edges();
faces = polytope.faces(2);
facets = polytope.facets();


## Print period sequences

def test_period_sequences():
    print("Period sequences:");
    print(period_series_truncated(7, [1,1,1,1,1,1], 5).coefficients(sparse = false));
    print(period_sequence(LG, 6));
    print(period_sequence(LG_mutated, 6));


## Compute the edge polynomials

def test_edge_polynomials():
    print("Edge polynomials:");
    edge_poly_list = [];
    for edge in edges:
        edge_polyhedron = Polyhedron(edge.vertices(), base_ring=ZZ);
        edge_polynomial = 0;
        integral_points_number = len(edge_polyhedron.integral_points());
        for i in range(integral_points_number):
            point = edge_polyhedron.integral_points()[i];
            monomial = R.monomial(*list(point));
            coeff = LG_mutated.monomial_coefficient(monomial);
            edge_polynomial += coeff*t^i;
        edge_poly_list.append(factor(edge_polynomial));
    print(edge_poly_list);


## Check that the chosen FRS triangulation
## actually defines a smooth toric variety

def test_triangulation_smoothness():
    triang_vertices = matrix(ZZ, [(0, -1, -1, -1, -1, -1, -1, -1, 0, 0, 0, 0, 0, 0, 1),
                                  (0, 0, 0, 0, 1, 1, 1, 2, -1, -1, 0, 1, 1, 2, 0),
                                  (0, 0, 0, 1, -1, 0, 0, -1, 0, 1, 0, 0, 0, -1, 0),
                                  (0, -1, 0, -1, 0, -1, 0, 0, 0, 0, 1, -1, 0, 0, 0)]);
    triang_simplices = matrix(ZZ, [(0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0),
                                   (1, 1, 1, 1, 1, 1, 1, 1, 2, 2, 2, 2, 2, 2, 2, 3, 3, 3, 3, 3, 3, 3, 4, 4, 4, 4, 4, 4, 4, 5, 6, 6, 7, 7, 8, 8, 8, 8, 9, 9, 10, 11),
                                   (2, 2, 2, 3, 3, 3, 4, 4, 3, 3, 3, 3, 4, 4, 8, 4, 5, 6, 6, 8, 9, 9, 5, 5, 6, 7, 7, 8, 8, 6, 7, 7, 10, 11, 9, 9, 10, 11, 10, 11, 12, 12),
                                   (3, 3, 4, 4, 5, 8, 5, 8, 4, 6, 8, 9, 6, 8, 9, 5, 6, 10, 11, 9, 10, 11, 6, 7, 7, 10, 11, 10, 11, 7, 10, 11, 12, 12, 10, 11, 13, 13, 12, 12, 13, 13),
                                   (4, 8, 8, 5, 11, 11, 11, 11, 6, 10, 9, 10, 10, 10, 10, 6, 11, 12, 12, 11, 12, 12, 7, 11, 10, 13, 13, 13, 13, 11, 12, 12, 13, 13, 14, 14, 14, 14, 14, 14, 14, 14)]);
    star_origin = 0;

    newton_polar = LatticePolytope(triang_vertices.columns());
    print("Check that the convex hull actually coincide with the dual polytope.");
    assert newton_polar.polyhedron() == polytope.polar().polyhedron()   
    pointConf = PointConfiguration(triang_vertices.columns(), star = star_origin);
    triangulation = Triangulation(triang_simplices.columns(), pointConf);
    print("Check that the triangulation is star with respect to the origin.");
    assert all(star_origin in simplex for simplex in triangulation);
    print("Check that the triangulation is regular.");
    assert triangulation.normal_cone().dim() > 0
    print("Is the associated toric variety smooth:");
    print(ToricVariety(triangulation.fan()).is_smooth());


## Check that all irreducible 2d Minkowski summands
## are hollow triangles of lattice width 1

def test_2d_minkowski_summands():
    bad = 0;
    for [P,Q,S] in face_minkowski_polytopes(LG_mutated, 2, refined = False):
        if (P.dim() != 2) : continue;
        if (len(P.vertices()) != 3) :
            bad = 1; break;
        if (len(P.interior_points()) > 0) :
            bad = 1; break;
        # We want to exclude the hollow triangle of lattice width 2
        if (P.polyhedron().volume() == 2) :
            if (max(P.facet_constants()) != 4):
                bad = 1; break;
    if not (bad):
        print("OK!");
    else:
        print("FAIL!");
    
    
## Check the smoothness of intersections of
## irreducible components of facet polynomials

def test_components_smoothness():
    bad = 0;
    for List in face_minkowski_polytopes(LG_mutated, 3, refined = True):
        L = len(List);
        par = List[0][-2].parent();
        T.<X_0, X_1, X_2> = toric_varieties.torus(3);

        for I in Subsets(range(L)):
            if (len(I) < 1) : continue;
            gens_list = [];
            component_list = [];

            for i in list(I): gens_list.append((List[i])[-2]);
            P = T.subscheme(par.ideal(gens_list).radical());
            
            # Check that irreducible components of the intersection are smooth
            for PP in P.irreducible_components():
                component_list.append(PP.defining_polynomials());
                II = par.ideal(PP.defining_polynomials()).radical();
                jacobian_list = [];
                for poly in II.gens():
                    gradient = [];
                    for g in par.gens(): gradient.append(poly.derivative(g));
                    jacobian_list.append(gradient);
                jacobian_matrix = matrix(par, jacobian_list);
                d = II.dimension();
                if (d < 1) : continue;
                c = 3 - d;
                if (c != len(I)) :
                    print("Warning: non-expected codimension.");
                    print("Codimension: " + str(c));
                    print("Number of generators of the radical: " + str(len(II.gens())));
                    print("Number of intersecting hypersurfaces: " + str(len(I)));
        
                singular_list = gens_list + jacobian_matrix.minors(c);
                singular_ideal = par.ideal(singular_list);
                if (singular_ideal.radical().gens().count(1) == 0):
                    bad = 1;

            # Check that the irreducible components of the intersection do not intersect
            for J in Subsets(range(len(component_list))):
                if (len(J) < 2) : continue;
                print("Warning: we actually have a reducible intersection.");
                intersection_list = [];
                for j in list(J): intersection_list += component_list[j];
                intersection_ideal = par.ideal(intersection_list);
                if (intersection_ideal.radical().gens().count(1) == 0):
                    bad = 1;

    if not (bad):
        print("OK!");
    else:
        print("FAIL!");


## For all facet polynomial print generators of ideals
## of intersections of its irreducible components
## while omitting linear generators and cases when
## there is only one non-linear generator
## (i.e., the only cases when the rationality
## over Q of these intersections is not immediate)

def test_components_rationality():
    bad = 0;
    for List in face_minkowski_polytopes(LG_mutated, 3, refined = True):
        L = len(List);
        par = List[0][-2].parent();
        par_poly = par.polynomial_ring();
        frac_field = FractionField(par_poly);
        T.<X_0, X_1, X_2> = toric_varieties.torus(3);

        for I in Subsets(range(L)):
            if (len(I) < 2) : continue;
            gens_list = [];
            component_list = [];

            for i in list(I): gens_list.append((List[i])[-2]);
            P = T.subscheme(par.ideal(gens_list).radical());
           
            # Check that irreducible components of the intersection are rational over Q
            for PP in P.irreducible_components():
                component_list.append(PP.defining_polynomials());
                II = par.ideal(PP.defining_polynomials()).radical();
                reduced_list = [];
                for poly in II.gens():
                    poly_reduced = par_poly(frac_field(poly).numerator());
                    if (poly_reduced.degree() != 1) : reduced_list.append(poly);
                if (len(reduced_list) > 1): print(reduced_list);

            # Check that the irreducible components of the intersection do not intersect
            for J in Subsets(range(len(component_list))):
                if (len(J) < 2) : continue;
                print("Warning: we actually have a reducible intersection.");
                intersection_list = [];
                for j in list(J): intersection_list += component_list[j];
                intersection_ideal = par.ideal(intersection_list);
                if (intersection_ideal.radical().gens().count(1) == 0):
                    print("Warning: components of the intersection are not disjoint.");


## Present facet polynomials in the form
## F(X_0, X_1) * X_2 + G(X_0, X_1) = 0,
## (it automatically checks that their Newton polytopes are
## of lattice width one), and check that components of
## F(X_0, X_1) = G(X_0, X_1) = 0 are defined over Q.

def test_boundary_rationality():
    for [P,Q,R] in face_minkowski_polytopes(LG_mutated, 3, refined = False):
        par = Q.parent();
        par_poly = par.polynomial_ring();
        frac_field = FractionField(par_poly);
        poly_reduced = par_poly(frac_field(Q).numerator());
        
        if (newton_polytope(Q).dim() != 3) : continue;
        output = linearize_width_one(Q)
        F = output['F']; G = output['G'];
        F = par_poly(frac_field(F).numerator());
        G = par_poly(frac_field(G).numerator());

        are_all_generators_linear = 1;
                       
        for J in par_poly.ideal([F,G]).radical().primary_decomposition():
            are_generators_linear = 1;
            for gen in J.gens():
                if (gen.degree() != 1) :
                    are_generators_linear = 0;
                    break;
            if not (are_generators_linear):
                print(J.gens());
                are_all_generators_linear = 0;

        if not (are_all_generators_linear) :
            print("Warning! Not all generators are linear.");
        else:
            print("OK!");


## Try a smooth fine regular star triangulation
## by bruteforcing the nearest mutations

#find_smooth_FRST(LG_mutated, 0)


## Compute facet polynomials

#face_polynomial(LG_mutated, 3)


## Tests

#test_period_sequences()
#test_edge_polynomials()
#test_triangulation_smoothness()
#test_2d_minkowski_summands()
#test_components_smoothness()
#test_components_rationality()
#test_boundary_rationality()